In [1]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
from datetime import datetime, timezone
from io import BytesIO
from uuid import uuid4

import hashlib
import sqlite3
import pandas as pd

phase3_database_path = Path(
    "../data/prototype/refresh_testing/monthly_refresh_fresh_test.db"
).resolve()

if not phase3_database_path.is_file():
    raise FileNotFoundError(
        f"Phase 3 test database not found: {phase3_database_path}"
    )

print("Phase 3 baseline found:", phase3_database_path)

Phase 3 baseline found: C:\Missed Refund Detection and Process Improvement\data\prototype\refresh_testing\monthly_refresh_fresh_test.db


In [2]:
phase4_folder = Path(
    "../data/prototype/accounting_testing"
).resolve()

phase4_folder.mkdir(parents=True, exist_ok=True)

phase4_database_path = phase4_folder / "accounting_check_test.db"

if phase4_database_path.exists():
    raise FileExistsError(
        "Phase 4 test database already exists. "
        "Stop here to avoid overwriting test work."
    )

with sqlite3.connect(
    phase3_database_path.as_uri() + "?mode=ro",
    uri=True,
) as baseline_connection:
    phase4_connection = sqlite3.connect(phase4_database_path)

    try:
        baseline_connection.backup(phase4_connection)
    finally:
        phase4_connection.close()

connection = sqlite3.connect(
    phase4_database_path.as_uri() + "?mode=rw",
    uri=True,
)

connection.execute("PRAGMA foreign_keys = ON;")

pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_cases,
        SUM(
            CASE WHEN "Case Status" = 'Completed'
            THEN 1 ELSE 0 END
        ) AS completed_cases,
        SUM(
            CASE WHEN "Case Status" <> 'Completed'
            THEN 1 ELSE 0 END
        ) AS unresolved_cases
    FROM agent_updates;
    """,
    connection,
)

,total_cases,completed_cases,unresolved_cases
0,2873,501,2372


In [3]:
case_population = pd.read_sql_query(
    """
    SELECT
        source.*,
        work."Case Status",
        work."Completed By"
    FROM source_cases AS source
    LEFT JOIN agent_updates AS work
        ON source."Case ID" = work."Case ID";
    """,
    connection,
)

if case_population["Case ID"].duplicated().any():
    raise ValueError("Duplicate case records detected.")

allowed_statuses = {
    "Open",
    "Awaiting Dependency",
    "Awaiting Senior Review",
    "Awaiting Other Department",
    "Completed",
}

if not case_population["Case Status"].isin(allowed_statuses).all():
    raise ValueError("Missing or unrecognised operational status.")

case_population["First Identified Date"] = pd.to_datetime(
    case_population["First Identified Date"],
    errors="raise",
)

if case_population["First Identified Date"].isna().any():
    raise ValueError("A case is missing its identification date.")

eligible_cases = case_population.loc[
    case_population["Case Status"].ne("Completed")
].copy()

print("Eligible unresolved cases:", len(eligible_cases))
print(
    "Completed cases excluded:",
    case_population["Case Status"].eq("Completed").sum(),
)
print(
    "Previously automated completions excluded:",
    (
        case_population["Case Status"].eq("Completed")
        & case_population["Completed By"].eq("Automated Check")
    ).sum(),
)

Eligible unresolved cases: 2372
Completed cases excluded: 501
Previously automated completions excluded: 500


In [4]:
ACCOUNTING_EXTRACT_DATE = pd.Timestamp("2026-08-31")

test_scenarios = [
    "Exact match",
    "Amount mismatch",
    "Refund before identification",
    "Refund on identification date",
    "Multiple valid matches",
    "No transaction",
    "Refund after extract date",
    "Refund on extract date",
    "Wrong client identifier",
    "Amount and date match different transactions",
]

accounting_test_cases = (
    eligible_cases
    .sort_values("Case ID")
    .head(len(test_scenarios))
    .copy()
    .reset_index(drop=True)
)

if len(accounting_test_cases) != len(test_scenarios):
    raise ValueError("Not enough unresolved cases for the tests.")

accounting_test_cases["Test Scenario"] = test_scenarios

if not (
    accounting_test_cases["First Identified Date"]
    < ACCOUNTING_EXTRACT_DATE
).all():
    raise ValueError("Test cases must predate the accounting extract.")

accounting_test_cases[
    ["Case ID", "First Identified Date", "Test Scenario"]
]

,Case ID,First Identified Date,Test Scenario
0,DH000001,2025-01-31,Exact match
1,DH000002,2025-01-31,Amount mismatch
2,DH000003,2025-01-31,Refund before identification
3,DH000004,2025-01-31,Refund on identification date
4,DH000005,2025-01-31,Multiple valid matches
5,DH000006,2025-01-31,No transaction
6,DH000007,2025-01-31,Refund after extract date
7,DH000008,2025-01-31,Refund on extract date
8,DH000009,2025-01-31,Wrong client identifier
9,DH000010,2025-01-31,Amount and date match different transactions


In [5]:
july_case_ids = {"TEST-JUL-001", "TEST-JUL-002"}

older_test_cases = (
    eligible_cases.loc[
        ~eligible_cases["Case ID"].isin(july_case_ids)
    ]
    .sort_values("Case ID")
    .head(8)
)

new_july_cases = (
    eligible_cases.loc[
        eligible_cases["Case ID"].isin(july_case_ids)
    ]
    .sort_values("Case ID")
)

if len(older_test_cases) != 8 or len(new_july_cases) != 2:
    raise ValueError("Expected eight older cases and two new July cases.")

accounting_test_cases = pd.concat(
    [older_test_cases, new_july_cases],
    ignore_index=True,
)

accounting_test_cases["Test Scenario"] = test_scenarios

if not (
    accounting_test_cases["First Identified Date"]
    < ACCOUNTING_EXTRACT_DATE
).all():
    raise ValueError("Test cases must predate the accounting extract.")

accounting_test_cases[
    ["Case ID", "First Identified Date", "Test Scenario"]
]

,Case ID,First Identified Date,Test Scenario
0,DH000001,2025-01-31,Exact match
1,DH000002,2025-01-31,Amount mismatch
2,DH000003,2025-01-31,Refund before identification
3,DH000004,2025-01-31,Refund on identification date
4,DH000005,2025-01-31,Multiple valid matches
5,DH000006,2025-01-31,No transaction
6,DH000007,2025-01-31,Refund after extract date
7,DH000008,2025-01-31,Refund on extract date
8,TEST-JUL-001,2026-07-31,Wrong client identifier
9,TEST-JUL-002,2026-07-31,Amount and date match different transactions


In [6]:
test_transaction_rows = []

def add_test_transaction(
    case,
    processed_date,
    amount=None,
    client_number=None,
):
    transaction_number = len(test_transaction_rows) + 1

    test_transaction_rows.append({
        "Refund Transaction ID": f"TEST-RT-{transaction_number:03d}",
        "Policy Number": case["Policy Number"],
        "Client Number": (
            case["Client Number"]
            if client_number is None
            else client_number
        ),
        "Refund Amount": str(
            Decimal(str(case["Outstanding Amount"]))
            if amount is None
            else Decimal(str(amount))
        ),
        "Refund Processed By": "TEST-FIN001",
        "Refund Processed Date": (
            pd.Timestamp(processed_date).strftime("%Y-%m-%d")
        ),
        "Refund Reason": (
            f"Synthetic test: {case['Test Scenario']}"
        ),
    })

print("Test transaction helper is ready.")

Test transaction helper is ready.


In [7]:
# Reset the list so rerunning this cell does not duplicate transactions.
test_transaction_rows.clear()

for _, case in accounting_test_cases.iterrows():
    scenario = case["Test Scenario"]
    identified_date = case["First Identified Date"]
    after_identification = identified_date + pd.Timedelta(days=1)
    amount = Decimal(str(case["Outstanding Amount"]))

    if scenario == "Exact match":
        add_test_transaction(case, after_identification)

    elif scenario == "Amount mismatch":
        add_test_transaction(
            case,
            after_identification,
            amount=amount + Decimal("1.00"),
        )

    elif scenario == "Refund before identification":
        add_test_transaction(
            case,
            identified_date - pd.Timedelta(days=1),
        )

    elif scenario == "Refund on identification date":
        add_test_transaction(case, identified_date)

    elif scenario == "Multiple valid matches":
        add_test_transaction(case, after_identification)
        add_test_transaction(
            case,
            after_identification + pd.Timedelta(days=1),
        )

    elif scenario == "No transaction":
        continue

    elif scenario == "Refund after extract date":
        add_test_transaction(
            case,
            ACCOUNTING_EXTRACT_DATE + pd.Timedelta(days=1),
        )

    elif scenario == "Refund on extract date":
        add_test_transaction(case, ACCOUNTING_EXTRACT_DATE)

    elif scenario == "Wrong client identifier":
        add_test_transaction(
            case,
            after_identification,
            client_number="TEST-WRONG-CLIENT",
        )

    elif scenario == "Amount and date match different transactions":
        add_test_transaction(case, identified_date)
        add_test_transaction(
            case,
            after_identification,
            amount=amount + Decimal("1.00"),
        )

    else:
        raise ValueError(f"Unknown test scenario: {scenario}")

test_accounting_feed = pd.DataFrame(test_transaction_rows)

assert len(test_accounting_feed) == 11
assert not test_accounting_feed[
    "Refund Transaction ID"
].duplicated().any()

print("Synthetic transactions:", len(test_accounting_feed))
print(
    "Transactions after extract date:",
    (
        pd.to_datetime(test_accounting_feed["Refund Processed Date"])
        > ACCOUNTING_EXTRACT_DATE
    ).sum(),
)

Synthetic transactions: 11
Transactions after extract date: 1


In [8]:
candidate_matches = accounting_test_cases.merge(
    test_accounting_feed,
    on=["Policy Number", "Client Number"],
    how="left",
    validate="one_to_many",
    indicator=True,
)

candidate_matches["Transaction Found"] = (
    candidate_matches["_merge"].eq("both")
)

candidate_matches["Amount Matches"] = candidate_matches.apply(
    lambda row: (
        Decimal(str(row["Outstanding Amount"]))
        == Decimal(str(row["Refund Amount"]))
        if row["Transaction Found"]
        else False
    ),
    axis=1,
)

processed_dates = pd.to_datetime(
    candidate_matches["Refund Processed Date"],
    errors="raise",
)

candidate_matches["Processed After Identification"] = (
    candidate_matches["Transaction Found"]
    & processed_dates.gt(
        candidate_matches["First Identified Date"]
    )
)

candidate_matches["Processed By Extract Date"] = (
    candidate_matches["Transaction Found"]
    & processed_dates.le(ACCOUNTING_EXTRACT_DATE)
)

candidate_matches["Valid Transaction Match"] = (
    candidate_matches["Transaction Found"]
    & candidate_matches["Amount Matches"]
    & candidate_matches["Processed After Identification"]
    & candidate_matches["Processed By Extract Date"]
)

print(
    "Transactions found:",
    candidate_matches["Transaction Found"].sum(),
)
print(
    "Valid transaction matches:",
    candidate_matches["Valid Transaction Match"].sum(),
)

Transactions found: 10
Valid transaction matches: 4


In [9]:
case_decisions = (
    candidate_matches
    .groupby("Case ID", as_index=False)
    .agg(
        Transactions_Found=("Transaction Found", "sum"),
        Valid_Match_Count=("Valid Transaction Match", "sum"),
    )
)

def choose_decision(valid_match_count):
    if valid_match_count == 1:
        return "Auto Complete"
    if valid_match_count > 1:
        return "Ambiguous Match"
    return "Agent Review"

case_decisions["Decision"] = (
    case_decisions["Valid_Match_Count"].map(choose_decision)
)

case_decisions = accounting_test_cases[
    ["Case ID", "Test Scenario"]
].merge(
    case_decisions,
    on="Case ID",
    validate="one_to_one",
)

expected_decisions = {
    "Exact match": "Auto Complete",
    "Amount mismatch": "Agent Review",
    "Refund before identification": "Agent Review",
    "Refund on identification date": "Agent Review",
    "Multiple valid matches": "Ambiguous Match",
    "No transaction": "Agent Review",
    "Refund after extract date": "Agent Review",
    "Refund on extract date": "Auto Complete",
    "Wrong client identifier": "Agent Review",
    "Amount and date match different transactions": "Agent Review",
}

assert case_decisions["Decision"].eq(
    case_decisions["Test Scenario"].map(expected_decisions)
).all(), "A scenario produced an unexpected decision."

print(case_decisions["Decision"].value_counts().to_string())
print("\nAll 10 scenario decisions match expectations.")

Decision
Agent Review       7
Auto Complete      2
Ambiguous Match    1

All 10 scenario decisions match expectations.


In [10]:
def explain_case_result(group):
    valid_count = int(group["Valid Transaction Match"].sum())

    if valid_count == 1:
        return "Exact Match"

    if valid_count > 1:
        return "Multiple Valid Matches"

    if not group["Transaction Found"].any():
        return "No Accounting Transaction"

    amount_matches = group.loc[group["Amount Matches"]]

    if amount_matches.empty:
        return "Amount Mismatch"

    after_identification = amount_matches.loc[
        amount_matches["Processed After Identification"]
    ]

    if after_identification.empty:
        return "Refund Not After Identification"

    if not after_identification["Processed By Extract Date"].any():
        return "Refund After Extract Date"

    raise ValueError("Unclassified accounting result.")

reason_rows = [
    {
        "Case ID": case_id,
        "Result Reason": explain_case_result(group),
    }
    for case_id, group in candidate_matches.groupby("Case ID")
]

case_decisions = case_decisions.drop(
    columns=["Result Reason"],
    errors="ignore",
).merge(
    pd.DataFrame(reason_rows),
    on="Case ID",
    validate="one_to_one",
)

case_decisions[
    ["Case ID", "Test Scenario", "Decision", "Result Reason"]
]

,Case ID,Test Scenario,Decision,Result Reason
0,DH000001,Exact match,Auto Complete,Exact Match
1,DH000002,Amount mismatch,Agent Review,Amount Mismatch
2,DH000003,Refund before identification,Agent Review,Refund Not After Identification
3,DH000004,Refund on identification date,Agent Review,Refund Not After Identification
4,DH000005,Multiple valid matches,Ambiguous Match,Multiple Valid Matches
5,DH000006,No transaction,Agent Review,No Accounting Transaction
6,DH000007,Refund after extract date,Agent Review,Refund After Extract Date
7,DH000008,Refund on extract date,Auto Complete,Exact Match
8,TEST-JUL-001,Wrong client identifier,Agent Review,No Accounting Transaction
9,TEST-JUL-002,Amount and date match different transactions,Agent Review,Refund Not After Identification


In [11]:
test_accounting_path = (
    phase4_folder / "refund_transactions_test.csv"
)

if test_accounting_path.exists():
    raise FileExistsError(
        "The accounting test file already exists."
    )

test_accounting_feed.to_csv(
    test_accounting_path,
    index=False,
)

accounting_file_bytes = test_accounting_path.read_bytes()

accounting_file_hash = hashlib.sha256(
    accounting_file_bytes
).hexdigest()

print("Saved:", test_accounting_path)
print("Transactions saved:", len(test_accounting_feed))
print("Accounting extract date:", ACCOUNTING_EXTRACT_DATE.date())

Saved: C:\Missed Refund Detection and Process Improvement\data\prototype\accounting_testing\refund_transactions_test.csv
Transactions saved: 11
Accounting extract date: 2026-08-31


In [12]:
connection.execute(
    """
    CREATE TABLE IF NOT EXISTS automation_runs (
        automation_run_id TEXT PRIMARY KEY,
        accounting_extract_date TEXT NOT NULL,
        source_file_name TEXT NOT NULL,
        source_file_hash TEXT NOT NULL,
        rule_version TEXT NOT NULL,
        started_at TEXT NOT NULL,
        completed_at TEXT,
        run_status TEXT NOT NULL
            CHECK (
                run_status IN (
                    'Started', 'Succeeded', 'Rejected', 'Failed'
                )
            ),
        eligible_case_count INTEGER NOT NULL DEFAULT 0,
        completed_cases_excluded INTEGER NOT NULL DEFAULT 0,
        cases_checked INTEGER NOT NULL DEFAULT 0,
        exact_match_count INTEGER NOT NULL DEFAULT 0,
        agent_review_count INTEGER NOT NULL DEFAULT 0,
        ambiguous_match_count INTEGER NOT NULL DEFAULT 0,
        updates_applied INTEGER NOT NULL DEFAULT 0,
        updates_skipped INTEGER NOT NULL DEFAULT 0,
        error_details TEXT
    );
    """
)

connection.commit()

print("Automation-run log is ready.")

Automation-run log is ready.


In [13]:
connection.execute(
    """
    CREATE TABLE IF NOT EXISTS automation_results (
        automation_run_id TEXT NOT NULL,
        case_id TEXT NOT NULL,
        checked_at TEXT NOT NULL,
        decision TEXT NOT NULL
            CHECK (
                decision IN (
                    'Auto Complete',
                    'Agent Review',
                    'Ambiguous Match'
                )
            ),
        result_reason TEXT NOT NULL,
        candidate_match_count INTEGER NOT NULL,
        valid_match_count INTEGER NOT NULL,
        expected_amount_pence INTEGER NOT NULL,
        first_identified_date TEXT NOT NULL,
        matched_transaction_id TEXT,
        matched_amount_pence INTEGER,
        refund_processed_date TEXT,
        completion_approved INTEGER NOT NULL
            CHECK (completion_approved IN (0, 1)),
        case_update_applied INTEGER NOT NULL DEFAULT 0
            CHECK (case_update_applied IN (0, 1)),
        update_applied_at TEXT,
        PRIMARY KEY (automation_run_id, case_id),
        FOREIGN KEY (automation_run_id)
            REFERENCES automation_runs (automation_run_id),
        FOREIGN KEY (case_id)
            REFERENCES source_cases ("Case ID"),
        CHECK (
            completion_approved = 0
            OR (
                decision = 'Auto Complete'
                AND result_reason = 'Exact Match'
                AND valid_match_count = 1
                AND matched_transaction_id IS NOT NULL
                AND matched_amount_pence = expected_amount_pence
                AND refund_processed_date IS NOT NULL
            )
        ),
        CHECK (
            case_update_applied = 0
            OR (
                completion_approved = 1
                AND update_applied_at IS NOT NULL
            )
        )
    );
    """
)

connection.commit()

print("Case-level automation results table is ready.")

Case-level automation results table is ready.


In [14]:
connection.execute(
    """
    CREATE TABLE IF NOT EXISTS workflow_events (
        event_id TEXT PRIMARY KEY,
        case_id TEXT NOT NULL,
        event_type TEXT NOT NULL,
        event_timestamp TEXT NOT NULL,
        actor_type TEXT NOT NULL,
        previous_status TEXT NOT NULL,
        new_status TEXT NOT NULL,
        previous_outcome TEXT,
        new_outcome TEXT,
        previous_completed_by TEXT,
        new_completed_by TEXT,
        previous_completion_date TEXT,
        new_completion_date TEXT,
        event_reason TEXT NOT NULL,
        automation_run_id TEXT NOT NULL,
        UNIQUE (automation_run_id, case_id, event_type),
        FOREIGN KEY (case_id)
            REFERENCES source_cases ("Case ID"),
        FOREIGN KEY (automation_run_id)
            REFERENCES automation_runs (automation_run_id)
    );
    """
)

connection.commit()

print("Automation completion-event history is ready.")

Automation completion-event history is ready.


In [15]:
def amount_to_pence(value):
    amount = Decimal(str(value))

    if (
        not amount.is_finite()
        or amount <= 0
        or amount != amount.quantize(Decimal("0.01"))
    ):
        raise ValueError(f"Invalid monetary amount: {value}")

    return int(amount * 100)


checked_at = datetime.now(timezone.utc).isoformat()
audit_result_rows = []

for _, result in case_decisions.iterrows():
    case_id = result["Case ID"]

    case = accounting_test_cases.loc[
        accounting_test_cases["Case ID"].eq(case_id)
    ].iloc[0]

    valid_transactions = candidate_matches.loc[
        candidate_matches["Case ID"].eq(case_id)
        & candidate_matches["Valid Transaction Match"]
    ]

    approved = result["Decision"] == "Auto Complete"

    matched = None

    if approved:
        if len(valid_transactions) != 1:
            raise ValueError("Approval requires exactly one valid match.")

        matched = valid_transactions.iloc[0]

    audit_result_rows.append({
        "case_id": case_id,
        "checked_at": checked_at,
        "decision": result["Decision"],
        "result_reason": result["Result Reason"],
        "candidate_match_count": int(result["Transactions_Found"]),
        "valid_match_count": int(result["Valid_Match_Count"]),
        "expected_amount_pence": amount_to_pence(
            case["Outstanding Amount"]
        ),
        "first_identified_date": (
            case["First Identified Date"].strftime("%Y-%m-%d")
        ),
        "matched_transaction_id": (
            matched["Refund Transaction ID"] if approved else None
        ),
        "matched_amount_pence": (
            amount_to_pence(matched["Refund Amount"])
            if approved else None
        ),
        "refund_processed_date": (
            pd.Timestamp(matched["Refund Processed Date"]).strftime(
                "%Y-%m-%d"
            )
            if approved else None
        ),
        "completion_approved": int(approved),
    })

assert len(audit_result_rows) == 10
assert sum(row["completion_approved"] for row in audit_result_rows) == 2

print("Audit results prepared:", len(audit_result_rows))
print("Approved completions:", 2)
print("Database case records remain unchanged.")

Audit results prepared: 10
Approved completions: 2
Database case records remain unchanged.


In [16]:
automation_run_id = f"TEST-AUTO-{uuid4().hex}"
RULE_VERSION = "first-identified-v1"

before_automation_updates = pd.read_sql_query(
    'SELECT * FROM agent_updates ORDER BY "Case ID";',
    connection,
)

result_columns = list(audit_result_rows[0])
column_sql = ", ".join(result_columns)
parameter_sql = ", ".join(f":{column}" for column in result_columns)

with connection:
    connection.execute(
        """
        INSERT INTO automation_runs (
            automation_run_id,
            accounting_extract_date,
            source_file_name,
            source_file_hash,
            rule_version,
            started_at,
            run_status,
            eligible_case_count,
            completed_cases_excluded,
            cases_checked,
            exact_match_count,
            agent_review_count,
            ambiguous_match_count
        )
        VALUES (?, ?, ?, ?, ?, ?, 'Started', 10, 0, 10, 2, 7, 1);
        """,
        (
            automation_run_id,
            ACCOUNTING_EXTRACT_DATE.strftime("%Y-%m-%d"),
            test_accounting_path.name,
            accounting_file_hash,
            RULE_VERSION,
            checked_at,
        ),
    )

    connection.executemany(
        f"""
        INSERT INTO automation_results (
            automation_run_id, {column_sql}
        )
        VALUES (:automation_run_id, {parameter_sql});
        """,
        [
            {"automation_run_id": automation_run_id, **row}
            for row in audit_result_rows
        ],
    )

print("Test run saved:", automation_run_id)
print("Case-level results saved:", len(audit_result_rows))
print("Completions applied: 0")

Test run saved: TEST-AUTO-1ea9ebe1e4f54537abe80eaa4ca9c977
Case-level results saved: 10
Completions applied: 0


In [17]:
approved_applications = []

for result in audit_result_rows:
    if not result["completion_approved"]:
        continue

    case_id = result["case_id"]

    source_at_check = accounting_test_cases.loc[
        accounting_test_cases["Case ID"].eq(case_id)
    ].iloc[0]

    work_at_check = before_automation_updates.loc[
        before_automation_updates["Case ID"].eq(case_id)
    ]

    if len(work_at_check) != 1:
        raise ValueError(f"Expected one operational record: {case_id}")

    work_at_check = work_at_check.iloc[0]

    if work_at_check["Case Status"] == "Completed":
        raise ValueError(f"Case is already completed: {case_id}")

    approved_applications.append({
        **result,
        "policy_number": source_at_check["Policy Number"],
        "client_number": source_at_check["Client Number"],
        "previous_status": work_at_check["Case Status"],
        "previous_outcome": work_at_check["Final Outcome"],
        "previous_completed_by": work_at_check["Completed By"],
        "previous_completion_date": work_at_check["Completion Date"],
        "previous_last_updated": work_at_check["Last Updated Date"],
        "previous_agent": work_at_check["Agent Working"],
        "previous_notes": work_at_check["Agent Notes"],
    })

assert len(approved_applications) == 2

print(
    "Approved cases:",
    [item["case_id"] for item in approved_applications],
)
print("Operational values captured for conflict checks.")

Approved cases: ['DH000001', 'DH000008']
Operational values captured for conflict checks.


In [18]:
def apply_guarded_completion(db_connection, run_id, item, applied_at):
    def sql_value(value):
        return None if pd.isna(value) else value

    parameters = {
        "case_id": item["case_id"],
        "run_id": run_id,
        "completion_date": item["refund_processed_date"],
        "applied_at": applied_at,
        "previous_status": item["previous_status"],
        "previous_outcome": sql_value(item["previous_outcome"]),
        "previous_completed_by": sql_value(item["previous_completed_by"]),
        "previous_completion_date": sql_value(
            item["previous_completion_date"]
        ),
        "previous_last_updated": sql_value(item["previous_last_updated"]),
        "previous_agent": sql_value(item["previous_agent"]),
        "previous_notes": sql_value(item["previous_notes"]),
        "policy_number": item["policy_number"],
        "client_number": item["client_number"],
        "expected_amount": item["expected_amount_pence"] / 100,
        "identified_date": item["first_identified_date"],
        "transaction_id": item["matched_transaction_id"],
    }

    cursor = db_connection.execute(
        """
        UPDATE agent_updates
        SET "Case Status" = 'Completed',
            "Final Outcome" = 'Refund Already Processed',
            "Completed By" = 'Automated Check',
            "Completion Date" = :completion_date,
            "Last Updated Date" = :applied_at
        WHERE "Case ID" = :case_id
          AND "Case Status" <> 'Completed'
          AND "Case Status" IS :previous_status
          AND "Final Outcome" IS :previous_outcome
          AND "Completed By" IS :previous_completed_by
          AND "Completion Date" IS :previous_completion_date
          AND "Last Updated Date" IS :previous_last_updated
          AND "Agent Working" IS :previous_agent
          AND "Agent Notes" IS :previous_notes
          AND EXISTS (
              SELECT 1 FROM source_cases AS source
              WHERE source."Case ID" = :case_id
                AND source."Policy Number" = :policy_number
                AND source."Client Number" = :client_number
                AND source."Outstanding Amount" = :expected_amount
                AND date(source."First Identified Date") = :identified_date
          )
          AND EXISTS (
              SELECT 1
              FROM automation_results AS result
              JOIN automation_runs AS run
                ON run.automation_run_id = result.automation_run_id
              WHERE result.automation_run_id = :run_id
                AND result.case_id = :case_id
                AND result.completion_approved = 1
                AND result.case_update_applied = 0
                AND result.matched_transaction_id = :transaction_id
                AND run.run_status = 'Started'
          )
          AND NOT EXISTS (
              SELECT 1 FROM automation_results
              WHERE matched_transaction_id = :transaction_id
                AND case_update_applied = 1
          );
        """,
        parameters,
    )

    return cursor.rowcount == 1

In [19]:
applied_at = datetime.now(timezone.utc).isoformat()
applied_case_ids = []
skipped_case_ids = []

if connection.in_transaction:
    raise RuntimeError("Finish the existing transaction first.")

try:
    connection.execute("BEGIN IMMEDIATE;")

    run_status = connection.execute(
        """
        SELECT run_status FROM automation_runs
        WHERE automation_run_id = ?;
        """,
        (automation_run_id,),
    ).fetchone()

    if run_status != ("Started",):
        raise ValueError("This run is not available for application.")

    before_apply = pd.read_sql_query(
        'SELECT * FROM agent_updates ORDER BY "Case ID";',
        connection,
    )

    expected_after = before_apply.copy()

    for item in approved_applications:
        case_id = item["case_id"]

        if not apply_guarded_completion(
            connection, automation_run_id, item, applied_at
        ):
            skipped_case_ids.append(case_id)
            continue

        cursor = connection.execute(
            """
            UPDATE automation_results
            SET case_update_applied = 1,
                update_applied_at = ?
            WHERE automation_run_id = ?
              AND case_id = ?
              AND case_update_applied = 0;
            """,
            (applied_at, automation_run_id, case_id),
        )

        if cursor.rowcount != 1:
            raise ValueError("Could not record the applied completion.")

        connection.execute(
            """
            INSERT INTO workflow_events (
                event_id, case_id, event_type, event_timestamp,
                actor_type, previous_status, new_status,
                previous_outcome, new_outcome,
                previous_completed_by, new_completed_by,
                previous_completion_date, new_completion_date,
                event_reason, automation_run_id
            )
            VALUES (
                ?, ?, 'Automated Completion', ?, 'Automated Process',
                ?, 'Completed', ?, 'Refund Already Processed',
                ?, 'Automated Check', ?, ?, ?, ?
            );
            """,
            (
                f"EVENT-{uuid4().hex}",
                case_id,
                applied_at,
                item["previous_status"],
                None if pd.isna(item["previous_outcome"])
                else item["previous_outcome"],
                None if pd.isna(item["previous_completed_by"])
                else item["previous_completed_by"],
                None if pd.isna(item["previous_completion_date"])
                else item["previous_completion_date"],
                item["refund_processed_date"],
                f"Exact Match: {item['matched_transaction_id']}",
                automation_run_id,
            ),
        )

        mask = expected_after["Case ID"].eq(case_id)

        for column, value in {
            "Case Status": "Completed",
            "Final Outcome": "Refund Already Processed",
            "Completed By": "Automated Check",
            "Completion Date": item["refund_processed_date"],
            "Last Updated Date": applied_at,
        }.items():
            expected_after[column] = expected_after[column].astype("object")
            expected_after.loc[mask, column] = value

        applied_case_ids.append(case_id)

    after_apply = pd.read_sql_query(
        'SELECT * FROM agent_updates ORDER BY "Case ID";',
        connection,
    )

    pd.testing.assert_frame_equal(
        after_apply,
        expected_after,
        check_dtype=False,
        check_exact=True,
    )

    connection.execute(
        """
        UPDATE automation_runs
        SET run_status = 'Succeeded',
            completed_at = ?,
            updates_applied = ?,
            updates_skipped = ?
        WHERE automation_run_id = ?;
        """,
        (
            datetime.now(timezone.utc).isoformat(),
            len(applied_case_ids),
            len(skipped_case_ids),
            automation_run_id,
        ),
    )

    connection.commit()

except Exception:
    connection.rollback()
    raise

print("Applied:", applied_case_ids)
print("Skipped:", skipped_case_ids)
print("All other operational values are unchanged.")

Applied: ['DH000001', 'DH000008']
Skipped: []
All other operational values are unchanged.


In [20]:
run_summary = pd.read_sql_query(
    """
    SELECT
        run_status,
        cases_checked,
        exact_match_count,
        agent_review_count,
        ambiguous_match_count,
        updates_applied,
        updates_skipped
    FROM automation_runs
    WHERE automation_run_id = ?;
    """,
    connection,
    params=(automation_run_id,),
)

applied_result_count = connection.execute(
    """
    SELECT COUNT(*)
    FROM automation_results
    WHERE automation_run_id = ?
      AND case_update_applied = 1;
    """,
    (automation_run_id,),
).fetchone()[0]

event_count = connection.execute(
    """
    SELECT COUNT(*)
    FROM workflow_events
    WHERE automation_run_id = ?;
    """,
    (automation_run_id,),
).fetchone()[0]

completed_count = connection.execute(
    """
    SELECT COUNT(*)
    FROM agent_updates
    WHERE "Case Status" = 'Completed';
    """
).fetchone()[0]

assert applied_result_count == 2
assert event_count == 2
assert completed_count == 503

print("Applied results:", applied_result_count)
print("Completion events:", event_count)
print("Total completed cases:", completed_count)

run_summary

Applied results: 2
Completion events: 2
Total completed cases: 503


,run_status,cases_checked,exact_match_count,agent_review_count,ambiguous_match_count,updates_applied,updates_skipped
0,Succeeded,10,2,7,1,2,0


In [21]:
before_repeat_work = pd.read_sql_query(
    'SELECT * FROM agent_updates ORDER BY "Case ID";',
    connection,
)

before_repeat_results = pd.read_sql_query(
    """
    SELECT * FROM automation_results
    ORDER BY automation_run_id, case_id;
    """,
    connection,
)

before_repeat_events = pd.read_sql_query(
    "SELECT * FROM workflow_events ORDER BY event_id;",
    connection,
)

try:
    connection.execute("BEGIN IMMEDIATE;")

    repeat_outcomes = [
        apply_guarded_completion(
            connection,
            automation_run_id,
            item,
            datetime.now(timezone.utc).isoformat(),
        )
        for item in approved_applications
    ]

    assert repeat_outcomes == [False, False]

    for table_name, order_by, expected in [
        ("agent_updates", '"Case ID"', before_repeat_work),
        (
            "automation_results",
            "automation_run_id, case_id",
            before_repeat_results,
        ),
        ("workflow_events", "event_id", before_repeat_events),
    ]:
        actual = pd.read_sql_query(
            f'SELECT * FROM "{table_name}" ORDER BY {order_by};',
            connection,
        )

        pd.testing.assert_frame_equal(actual, expected)

finally:
    connection.rollback()

print("Both repeated completion attempts were blocked.")
print("Operational records, results and events are unchanged.")

Both repeated completion attempts were blocked.
Operational records, results and events are unchanged.


In [22]:
conflict_case_id = "DH000002"

conflict_source = pd.read_sql_query(
    'SELECT * FROM source_cases WHERE "Case ID" = ?;',
    connection,
    params=(conflict_case_id,),
).iloc[0]

conflict_work = pd.read_sql_query(
    'SELECT * FROM agent_updates WHERE "Case ID" = ?;',
    connection,
    params=(conflict_case_id,),
).iloc[0]

assert conflict_work["Case Status"] != "Completed"

conflict_item = {
    **audit_result_rows[0],
    "case_id": conflict_case_id,
    "expected_amount_pence": amount_to_pence(
        conflict_source["Outstanding Amount"]
    ),
    "first_identified_date": pd.Timestamp(
        conflict_source["First Identified Date"]
    ).strftime("%Y-%m-%d"),
    "matched_transaction_id": "TEST-CONFLICT-RT-001",
    "matched_amount_pence": amount_to_pence(
        conflict_source["Outstanding Amount"]
    ),
    "refund_processed_date": (
        pd.Timestamp(conflict_source["First Identified Date"])
        + pd.Timedelta(days=1)
    ).strftime("%Y-%m-%d"),
    "policy_number": conflict_source["Policy Number"],
    "client_number": conflict_source["Client Number"],
    "previous_status": conflict_work["Case Status"],
    "previous_outcome": conflict_work["Final Outcome"],
    "previous_completed_by": conflict_work["Completed By"],
    "previous_completion_date": conflict_work["Completion Date"],
    "previous_last_updated": conflict_work["Last Updated Date"],
    "previous_agent": conflict_work["Agent Working"],
    "previous_notes": conflict_work["Agent Notes"],
}

print("Conflict-test fixture prepared:", conflict_case_id)
print("Database records are unchanged.")

Conflict-test fixture prepared: DH000002
Database records are unchanged.


In [23]:
conflict_run_id = f"TEST-CONFLICT-{uuid4().hex}"
test_time = datetime.now(timezone.utc).isoformat()

tables_to_check = {
    "agent_updates": '"Case ID"',
    "automation_runs": "automation_run_id",
    "automation_results": "automation_run_id, case_id",
    "workflow_events": "event_id",
}

before_conflict_test = {
    table: pd.read_sql_query(
        f'SELECT * FROM "{table}" ORDER BY {order};',
        connection,
    )
    for table, order in tables_to_check.items()
}

if connection.in_transaction:
    raise RuntimeError("Finish the existing transaction first.")

try:
    connection.execute("BEGIN IMMEDIATE;")

    connection.execute(
        """
        INSERT INTO automation_runs (
            automation_run_id, accounting_extract_date,
            source_file_name, source_file_hash,
            rule_version, started_at, run_status
        )
        VALUES (?, ?, ?, ?, ?, ?, 'Started');
        """,
        (
            conflict_run_id,
            ACCOUNTING_EXTRACT_DATE.strftime("%Y-%m-%d"),
            "synthetic-conflict-fixture",
            accounting_file_hash,
            RULE_VERSION,
            test_time,
        ),
    )

    connection.execute(
        f"""
        INSERT INTO automation_results (
            automation_run_id, {column_sql}
        )
        VALUES (:automation_run_id, {parameter_sql});
        """,
        {"automation_run_id": conflict_run_id, **conflict_item},
    )

    # Positive control: the unchanged case must be eligible.
    connection.execute("SAVEPOINT unchanged_case;")

    assert apply_guarded_completion(
        connection, conflict_run_id, conflict_item, test_time
    ), "The unchanged case was unexpectedly blocked."

    connection.execute("ROLLBACK TO unchanged_case;")
    connection.execute("RELEASE unchanged_case;")

    # Simulate an agent saving a newer note after approval.
    connection.execute(
        """
        UPDATE agent_updates
        SET "Agent Notes" = ?,
            "Last Updated Date" = ?
        WHERE "Case ID" = ?;
        """,
        (
            "Synthetic newer agent note",
            test_time,
            conflict_case_id,
        ),
    )

    work_before_attempt = pd.read_sql_query(
        'SELECT * FROM agent_updates WHERE "Case ID" = ?;',
        connection,
        params=(conflict_case_id,),
    )

    assert not apply_guarded_completion(
        connection, conflict_run_id, conflict_item, test_time
    ), "A stale approval overwrote newer agent activity."

    pd.testing.assert_frame_equal(
        work_before_attempt,
        pd.read_sql_query(
            'SELECT * FROM agent_updates WHERE "Case ID" = ?;',
            connection,
            params=(conflict_case_id,),
        ),
    )

finally:
    connection.rollback()

for table, order in tables_to_check.items():
    pd.testing.assert_frame_equal(
        before_conflict_test[table],
        pd.read_sql_query(
            f'SELECT * FROM "{table}" ORDER BY {order};',
            connection,
        ),
    )

print("Unchanged case: completion allowed.")
print("Newer agent note: stale approval blocked.")
print("All conflict-test changes rolled back.")

Unchanged case: completion allowed.
Newer agent note: stale approval blocked.
All conflict-test changes rolled back.


In [24]:
conflict_run_id = f"TEST-CONFLICT-{uuid4().hex}"
test_time = datetime.now(timezone.utc).isoformat()

tables_to_check = {
    "agent_updates": '"Case ID"',
    "automation_runs": "automation_run_id",
    "automation_results": "automation_run_id, case_id",
    "workflow_events": "event_id",
}

if connection.in_transaction:
    raise RuntimeError("Finish the existing transaction first.")

before_conflict_test = {
    table: pd.read_sql_query(
        f'SELECT * FROM "{table}" ORDER BY {order};',
        connection,
    )
    for table, order in tables_to_check.items()
}

try:
    connection.execute("BEGIN IMMEDIATE;")

    connection.execute(
        """
        INSERT INTO automation_runs (
            automation_run_id, accounting_extract_date,
            source_file_name, source_file_hash,
            rule_version, started_at, run_status
        )
        VALUES (?, ?, ?, ?, ?, ?, 'Started');
        """,
        (
            conflict_run_id,
            ACCOUNTING_EXTRACT_DATE.strftime("%Y-%m-%d"),
            "synthetic-agent-completion-fixture",
            accounting_file_hash,
            RULE_VERSION,
            test_time,
        ),
    )

    connection.execute(
        f"""
        INSERT INTO automation_results (
            automation_run_id, {column_sql}
        )
        VALUES (:automation_run_id, {parameter_sql});
        """,
        {"automation_run_id": conflict_run_id, **conflict_item},
    )

    # Positive control: an unchanged case must be eligible.
    connection.execute("SAVEPOINT unchanged_case;")

    assert apply_guarded_completion(
        connection,
        conflict_run_id,
        conflict_item,
        test_time,
    ), "The unchanged case was unexpectedly blocked."

    connection.execute("ROLLBACK TO unchanged_case;")
    connection.execute("RELEASE unchanged_case;")

    # Simulate an agent completing the case after approval.
    connection.execute(
        """
        UPDATE agent_updates
        SET "Case Status" = 'Completed',
            "Final Outcome" = 'No Refund Due',
            "Completed By" = 'TEST-AGENT',
            "Completion Date" = ?,
            "Last Updated Date" = ?
        WHERE "Case ID" = ?;
        """,
        (
            pd.Timestamp(test_time).strftime("%Y-%m-%d"),
            test_time,
            conflict_case_id,
        ),
    )

    work_before_attempt = pd.read_sql_query(
        'SELECT * FROM agent_updates WHERE "Case ID" = ?;',
        connection,
        params=(conflict_case_id,),
    )

    assert not apply_guarded_completion(
        connection,
        conflict_run_id,
        conflict_item,
        test_time,
    ), "Automation overwrote a newer agent completion."

    work_after_attempt = pd.read_sql_query(
        'SELECT * FROM agent_updates WHERE "Case ID" = ?;',
        connection,
        params=(conflict_case_id,),
    )

    pd.testing.assert_frame_equal(
        work_before_attempt,
        work_after_attempt,
    )

    assert work_after_attempt.loc[0, "Case Status"] == "Completed"
    assert work_after_attempt.loc[0, "Final Outcome"] == "No Refund Due"
    assert work_after_attempt.loc[0, "Completed By"] == "TEST-AGENT"

finally:
    connection.rollback()

for table, order in tables_to_check.items():
    pd.testing.assert_frame_equal(
        before_conflict_test[table],
        pd.read_sql_query(
            f'SELECT * FROM "{table}" ORDER BY {order};',
            connection,
        ),
    )

print("Unchanged case: completion allowed.")
print("Agent completion: stale approval blocked.")
print("Agent outcome and completion details were preserved.")
print("All conflict-test changes rolled back.")

Unchanged case: completion allowed.
Agent completion: stale approval blocked.
Agent outcome and completion details were preserved.
All conflict-test changes rolled back.


In [25]:
rollback_run_id = f"TEST-ROLLBACK-{uuid4().hex}"
test_time = datetime.now(timezone.utc).isoformat()

tables_to_check = {
    "agent_updates": '"Case ID"',
    "automation_runs": "automation_run_id",
    "automation_results": "automation_run_id, case_id",
    "workflow_events": "event_id",
}

if connection.in_transaction:
    raise RuntimeError("Finish the existing transaction first.")

before_rollback_test = {
    table: pd.read_sql_query(
        f'SELECT * FROM "{table}" ORDER BY {order};',
        connection,
    )
    for table, order in tables_to_check.items()
}

expected_failure_seen = False

try:
    connection.execute("BEGIN IMMEDIATE;")

    connection.execute(
        """
        INSERT INTO automation_runs (
            automation_run_id, accounting_extract_date,
            source_file_name, source_file_hash,
            rule_version, started_at, run_status
        )
        VALUES (?, ?, ?, ?, ?, ?, 'Started');
        """,
        (
            rollback_run_id,
            ACCOUNTING_EXTRACT_DATE.strftime("%Y-%m-%d"),
            "synthetic-rollback-fixture",
            accounting_file_hash,
            RULE_VERSION,
            test_time,
        ),
    )

    connection.execute(
        f"""
        INSERT INTO automation_results (
            automation_run_id, {column_sql}
        )
        VALUES (:automation_run_id, {parameter_sql});
        """,
        {"automation_run_id": rollback_run_id, **conflict_item},
    )

    assert apply_guarded_completion(
        connection,
        rollback_run_id,
        conflict_item,
        test_time,
    ), "The test case was unexpectedly blocked."

    cursor = connection.execute(
        """
        UPDATE automation_results
        SET case_update_applied = 1,
            update_applied_at = ?
        WHERE automation_run_id = ?
          AND case_id = ?;
        """,
        (test_time, rollback_run_id, conflict_case_id),
    )

    assert cursor.rowcount == 1

    # Simulate failure before the completion event can be saved.
    raise RuntimeError("Intentional accounting rollback test")

except RuntimeError as exc:
    if str(exc) != "Intentional accounting rollback test":
        raise

    expected_failure_seen = True

finally:
    connection.rollback()

assert expected_failure_seen

for table, order in tables_to_check.items():
    pd.testing.assert_frame_equal(
        before_rollback_test[table],
        pd.read_sql_query(
            f'SELECT * FROM "{table}" ORDER BY {order};',
            connection,
        ),
    )

print("Intentional processing failure detected.")
print("Case completion and applied-result flag rolled back.")
print("No partial test run, result or history event remains.")
print("All four tables match their before-test state.")

Intentional processing failure detected.
Case completion and applied-result flag rolled back.
No partial test run, result or history event remains.
All four tables match their before-test state.


In [26]:
def load_accounting_extract(file_path):
    file_path = Path(file_path)
    file_bytes = file_path.read_bytes()

    feed = pd.read_csv(
        BytesIO(file_bytes),
        dtype="string",
        keep_default_na=False,
    )

    required_columns = [
        "Refund Transaction ID",
        "Policy Number",
        "Client Number",
        "Refund Amount",
        "Refund Processed By",
        "Refund Processed Date",
        "Refund Reason",
    ]

    missing = set(required_columns) - set(feed.columns)

    if missing:
        raise ValueError(
            f"Missing accounting columns: {sorted(missing)}"
        )

    feed = feed[required_columns].copy()

    for column in required_columns:
        feed[column] = feed[column].str.strip()

    mandatory_columns = [
        column
        for column in required_columns
        if column != "Refund Reason"
    ]

    if feed[mandatory_columns].eq("").any().any():
        raise ValueError("Accounting extract has blank required values.")

    if feed["Refund Transaction ID"].duplicated().any():
        raise ValueError("Duplicate refund transaction IDs.")

    feed["Refund Amount Pence"] = (
        feed["Refund Amount"].map(amount_to_pence)
    )

    feed["Refund Processed Date"] = pd.to_datetime(
        feed["Refund Processed Date"],
        format="%Y-%m-%d",
        errors="raise",
    )

    return {
        "feed": feed,
        "file_name": file_path.name,
        "file_hash": hashlib.sha256(file_bytes).hexdigest(),
    }

In [27]:
loaded_extract = load_accounting_extract(test_accounting_path)

assert len(loaded_extract["feed"]) == 11
assert loaded_extract["file_hash"] == accounting_file_hash

future_count = loaded_extract["feed"][
    "Refund Processed Date"
].gt(ACCOUNTING_EXTRACT_DATE).sum()

assert future_count == 1

print("Validated transactions:", len(loaded_extract["feed"]))
print("File fingerprint matches the saved test input.")
print("Future-dated transactions retained:", future_count)

Validated transactions: 11
File fingerprint matches the saved test input.
Future-dated transactions retained: 1


In [28]:
def assess_accounting_cases(cases, accounting_feed, extract_date):
    cases = cases.copy()
    extract_date = pd.Timestamp(extract_date)

    if pd.isna(extract_date):
        raise ValueError("Accounting extract date is required.")

    if cases["Case ID"].isna().any():
        raise ValueError("Missing Case IDs.")

    if cases["Case ID"].duplicated().any():
        raise ValueError("Duplicate Case IDs.")

    cases["First Identified Date"] = pd.to_datetime(
        cases["First Identified Date"],
        errors="raise",
    )

    if cases["First Identified Date"].isna().any():
        raise ValueError("Missing identification dates.")

    cases["Expected Amount Pence"] = (
        cases["Outstanding Amount"].map(amount_to_pence)
    )

    candidates = cases.merge(
        accounting_feed,
        on=["Policy Number", "Client Number"],
        how="left",
        validate="one_to_many",
        indicator=True,
    )

    candidates["Transaction Found"] = candidates["_merge"].eq("both")

    candidates["Amount Matches"] = (
        candidates["Transaction Found"]
        & candidates["Expected Amount Pence"].eq(
            candidates["Refund Amount Pence"]
        )
    )

    candidates["Processed After Identification"] = (
        candidates["Transaction Found"]
        & candidates["Refund Processed Date"].gt(
            candidates["First Identified Date"]
        )
    )

    candidates["Processed By Extract Date"] = (
        candidates["Transaction Found"]
        & candidates["Refund Processed Date"].le(extract_date)
    )

    candidates["Valid Transaction Match"] = (
        candidates["Transaction Found"]
        & candidates["Amount Matches"]
        & candidates["Processed After Identification"]
        & candidates["Processed By Extract Date"]
    )

    decisions = candidates.groupby("Case ID", as_index=False).agg(
        Transactions_Found=("Transaction Found", "sum"),
        Valid_Match_Count=("Valid Transaction Match", "sum"),
    )

    decisions["Decision"] = (
        decisions["Valid_Match_Count"].map(choose_decision)
    )

    reasons = {
        case_id: explain_case_result(group)
        for case_id, group in candidates.groupby("Case ID")
    }

    decisions["Result Reason"] = decisions["Case ID"].map(reasons)

    return decisions, candidates

In [29]:
reusable_decisions, reusable_candidates = assess_accounting_cases(
    accounting_test_cases,
    loaded_extract["feed"],
    ACCOUNTING_EXTRACT_DATE,
)

comparison_columns = [
    "Case ID",
    "Transactions_Found",
    "Valid_Match_Count",
    "Decision",
    "Result Reason",
]

pd.testing.assert_frame_equal(
    reusable_decisions[comparison_columns]
    .sort_values("Case ID")
    .reset_index(drop=True),
    case_decisions[comparison_columns]
    .sort_values("Case ID")
    .reset_index(drop=True),
    check_dtype=False,
)

print("Reusable matching reproduces all 10 scenario results.")

Reusable matching reproduces all 10 scenario results.


In [30]:
def load_eligible_cases(db_connection):
    population = pd.read_sql_query(
        """
        SELECT
            source.*,
            work."Case Status",
            work."Completed By"
        FROM source_cases AS source
        LEFT JOIN agent_updates AS work
            ON source."Case ID" = work."Case ID";
        """,
        db_connection,
    )

    if population["Case ID"].isna().any():
        raise ValueError("Missing source Case IDs.")

    if population["Case ID"].duplicated().any():
        raise ValueError("Duplicate case records.")

    allowed_statuses = {
        "Open",
        "Awaiting Dependency",
        "Awaiting Senior Review",
        "Awaiting Other Department",
        "Completed",
    }

    if not population["Case Status"].isin(allowed_statuses).all():
        raise ValueError("Missing or unrecognised operational status.")

    completed = population["Case Status"].eq("Completed")

    return (
        population.loc[~completed].copy(),
        int(completed.sum()),
    )

In [31]:
current_eligible_cases, excluded_completed_count = (
    load_eligible_cases(connection)
)

assert len(current_eligible_cases) == 2370
assert excluded_completed_count == 503

print("Current eligible cases:", len(current_eligible_cases))
print("Completed cases excluded:", excluded_completed_count)

Current eligible cases: 2370
Completed cases excluded: 503


In [32]:
def prepare_accounting_results(
    cases,
    decisions,
    candidates,
    work_records,
    checked_at,
):
    if work_records["Case ID"].duplicated().any():
        raise ValueError("Duplicate operational records.")

    if (
        decisions["Case ID"].duplicated().any()
        or set(decisions["Case ID"]) != set(cases["Case ID"])
    ):
        raise ValueError("Decisions do not match the eligible cases.")

    source_lookup = cases.set_index("Case ID")
    work_lookup = work_records.set_index("Case ID")

    previous_fields = {
        "previous_status": "Case Status",
        "previous_outcome": "Final Outcome",
        "previous_completed_by": "Completed By",
        "previous_completion_date": "Completion Date",
        "previous_last_updated": "Last Updated Date",
        "previous_agent": "Agent Working",
        "previous_notes": "Agent Notes",
    }

    audit_rows = []
    applications = []

    for _, decision in decisions.iterrows():
        case_id = decision["Case ID"]
        source = source_lookup.loc[case_id]
        work = work_lookup.loc[case_id]

        if work["Case Status"] == "Completed":
            raise ValueError("Completed case entered result preparation.")

        approved = decision["Decision"] == "Auto Complete"

        valid = candidates.loc[
            candidates["Case ID"].eq(case_id)
            & candidates["Valid Transaction Match"]
        ]

        if approved and len(valid) != 1:
            raise ValueError("Approval requires exactly one valid match.")

        match = valid.iloc[0] if approved else None

        result = {
            "case_id": case_id,
            "checked_at": checked_at,
            "decision": decision["Decision"],
            "result_reason": decision["Result Reason"],
            "candidate_match_count": int(decision["Transactions_Found"]),
            "valid_match_count": int(decision["Valid_Match_Count"]),
            "expected_amount_pence": amount_to_pence(
                source["Outstanding Amount"]
            ),
            "first_identified_date": pd.Timestamp(
                source["First Identified Date"]
            ).strftime("%Y-%m-%d"),
            "matched_transaction_id": (
                match["Refund Transaction ID"] if approved else None
            ),
            "matched_amount_pence": (
                int(match["Refund Amount Pence"]) if approved else None
            ),
            "refund_processed_date": (
                match["Refund Processed Date"].strftime("%Y-%m-%d")
                if approved else None
            ),
            "completion_approved": int(approved),
        }

        audit_rows.append(result)

        if approved:
            applications.append({
                **result,
                "policy_number": source["Policy Number"],
                "client_number": source["Client Number"],
                **{
                    key: work[column]
                    for key, column in previous_fields.items()
                },
            })

    return audit_rows, applications

In [33]:
current_decisions, current_candidates = assess_accounting_cases(
    current_eligible_cases,
    loaded_extract["feed"],
    ACCOUNTING_EXTRACT_DATE,
)

current_work = pd.read_sql_query(
    "SELECT * FROM agent_updates;",
    connection,
)

prepared_results, prepared_applications = prepare_accounting_results(
    current_eligible_cases,
    current_decisions,
    current_candidates,
    current_work,
    datetime.now(timezone.utc).isoformat(),
)

assert len(prepared_results) == 2370
assert len(prepared_applications) == 0

print("Audit results prepared:", len(prepared_results))
print("New completions approved:", len(prepared_applications))

Audit results prepared: 2370
New completions approved: 0


In [34]:
def save_accounting_results(db_connection, run_id, audit_rows):
    result_columns = [
        "case_id",
        "checked_at",
        "decision",
        "result_reason",
        "candidate_match_count",
        "valid_match_count",
        "expected_amount_pence",
        "first_identified_date",
        "matched_transaction_id",
        "matched_amount_pence",
        "refund_processed_date",
        "completion_approved",
    ]

    column_sql = ", ".join(result_columns)
    parameter_sql = ", ".join(
        f":{column}" for column in result_columns
    )

    db_connection.executemany(
        f"""
        INSERT INTO automation_results (
            automation_run_id, {column_sql}
        )
        VALUES (:automation_run_id, {parameter_sql});
        """,
        [
            {
                "automation_run_id": run_id,
                **{column: row[column] for column in result_columns},
            }
            for row in audit_rows
        ],
    )

    saved_count = db_connection.execute(
        """
        SELECT COUNT(*)
        FROM automation_results
        WHERE automation_run_id = ?;
        """,
        (run_id,),
    ).fetchone()[0]

    if saved_count != len(audit_rows):
        raise ValueError("Saved result count does not reconcile.")

    return saved_count


print("Reusable audit-result writer is ready.")

Reusable audit-result writer is ready.


In [35]:
def apply_accounting_approvals(db_connection, run_id, applications):
    applied_at = datetime.now(timezone.utc).isoformat()

    expected_work = pd.read_sql_query(
        'SELECT * FROM agent_updates ORDER BY "Case ID";',
        db_connection,
    )

    applied_ids = []
    skipped_ids = []

    def nullable(value):
        return None if pd.isna(value) else value

    for item in applications:
        case_id = item["case_id"]

        if not apply_guarded_completion(
            db_connection, run_id, item, applied_at
        ):
            skipped_ids.append(case_id)
            continue

        cursor = db_connection.execute(
            """
            UPDATE automation_results
            SET case_update_applied = 1,
                update_applied_at = ?
            WHERE automation_run_id = ?
              AND case_id = ?
              AND case_update_applied = 0;
            """,
            (applied_at, run_id, case_id),
        )

        if cursor.rowcount != 1:
            raise ValueError("Could not record the applied completion.")

        db_connection.execute(
            """
            INSERT INTO workflow_events (
                event_id, case_id, event_type, event_timestamp,
                actor_type, previous_status, new_status,
                previous_outcome, new_outcome,
                previous_completed_by, new_completed_by,
                previous_completion_date, new_completion_date,
                event_reason, automation_run_id
            )
            VALUES (
                ?, ?, 'Automated Completion', ?, 'Automated Process',
                ?, 'Completed', ?, 'Refund Already Processed',
                ?, 'Automated Check', ?, ?, ?, ?
            );
            """,
            (
                f"EVENT-{uuid4().hex}",
                case_id,
                applied_at,
                item["previous_status"],
                nullable(item["previous_outcome"]),
                nullable(item["previous_completed_by"]),
                nullable(item["previous_completion_date"]),
                item["refund_processed_date"],
                f"Exact Match: {item['matched_transaction_id']}",
                run_id,
            ),
        )

        changes = {
            "Case Status": "Completed",
            "Final Outcome": "Refund Already Processed",
            "Completed By": "Automated Check",
            "Completion Date": item["refund_processed_date"],
            "Last Updated Date": applied_at,
        }

        mask = expected_work["Case ID"].eq(case_id)

        for column, value in changes.items():
            expected_work[column] = expected_work[column].astype("object")
            expected_work.loc[mask, column] = value

        applied_ids.append(case_id)

    actual_work = pd.read_sql_query(
        'SELECT * FROM agent_updates ORDER BY "Case ID";',
        db_connection,
    )

    pd.testing.assert_frame_equal(
        actual_work,
        expected_work,
        check_dtype=False,
        check_exact=True,
    )

    return applied_ids, skipped_ids


print("Reusable completion-and-event writer is ready.")

Reusable completion-and-event writer is ready.


In [36]:
def run_accounting_check(
    file_path,
    extract_date,
    db_connection,
    run_id,
    rule_version="first-identified-v1",
):
    if db_connection.in_transaction:
        raise RuntimeError("Finish the existing transaction first.")

    file_path = Path(file_path)
    extract_date = pd.Timestamp(extract_date)

    if pd.isna(extract_date):
        raise ValueError("Accounting extract date is required.")

    extract_date_text = extract_date.strftime("%Y-%m-%d")
    file_hash = hashlib.sha256(file_path.read_bytes()).hexdigest()

    def summary():
        return pd.read_sql_query(
            """
            SELECT automation_run_id, run_status,
                   eligible_case_count, completed_cases_excluded,
                   cases_checked, exact_match_count,
                   agent_review_count, ambiguous_match_count,
                   updates_applied, updates_skipped, error_details
            FROM automation_runs
            WHERE automation_run_id = ?;
            """,
            db_connection,
            params=(run_id,),
        )

    existing = db_connection.execute(
        """
        SELECT source_file_hash, accounting_extract_date,
               rule_version, run_status
        FROM automation_runs
        WHERE automation_run_id = ?;
        """,
        (run_id,),
    ).fetchone()

    if existing is not None:
        if existing[:3] != (
            file_hash, extract_date_text, rule_version
        ):
            raise ValueError(
                "Run ID already belongs to different input or rules."
            )

        if existing[3] == "Succeeded":
            return summary()

        raise ValueError(
            "This run ID already exists but has not succeeded. "
            "Use a new attempt ID."
        )

    started_at = datetime.now(timezone.utc).isoformat()

    with db_connection:
        db_connection.execute(
            """
            INSERT INTO automation_runs (
                automation_run_id, accounting_extract_date,
                source_file_name, source_file_hash,
                rule_version, started_at, run_status
            )
            VALUES (?, ?, ?, ?, ?, ?, 'Started');
            """,
            (
                run_id, extract_date_text, file_path.name,
                file_hash, rule_version, started_at,
            ),
        )

    stage = "validation"

    try:
        loaded = load_accounting_extract(file_path)

        if loaded["file_hash"] != file_hash:
            raise ValueError("Input file changed during loading.")

        stage = "processing"
        db_connection.execute("BEGIN IMMEDIATE;")

        cases, excluded_count = load_eligible_cases(db_connection)

        work = pd.read_sql_query(
            "SELECT * FROM agent_updates;",
            db_connection,
        )

        if cases.empty:
            audit_rows, applications = [], []
        else:
            decisions, candidates = assess_accounting_cases(
                cases, loaded["feed"], extract_date
            )

            audit_rows, applications = prepare_accounting_results(
                cases,
                decisions,
                candidates,
                work,
                datetime.now(timezone.utc).isoformat(),
            )

        saved_count = save_accounting_results(
            db_connection, run_id, audit_rows
        )

        applied_ids, skipped_ids = apply_accounting_approvals(
            db_connection, run_id, applications
        )

        decision_counts = {
            name: sum(row["decision"] == name for row in audit_rows)
            for name in [
                "Auto Complete", "Agent Review", "Ambiguous Match"
            ]
        }

        if saved_count != len(cases):
            raise ValueError("Eligible and checked counts differ.")

        if sum(decision_counts.values()) != saved_count:
            raise ValueError("Decision counts do not reconcile.")

        if len(applied_ids) + len(skipped_ids) != len(applications):
            raise ValueError("Application counts do not reconcile.")

        applied_count, event_count = db_connection.execute(
            """
            SELECT
                (
                    SELECT COUNT(*) FROM automation_results
                    WHERE automation_run_id = ?
                      AND case_update_applied = 1
                ),
                (
                    SELECT COUNT(*) FROM workflow_events
                    WHERE automation_run_id = ?
                );
            """,
            (run_id, run_id),
        ).fetchone()

        if applied_count != len(applied_ids) or event_count != applied_count:
            raise ValueError("Applied results and events do not reconcile.")

        db_connection.execute(
            """
            UPDATE automation_runs
            SET run_status = 'Succeeded', completed_at = ?,
                eligible_case_count = ?,
                completed_cases_excluded = ?,
                cases_checked = ?, exact_match_count = ?,
                agent_review_count = ?, ambiguous_match_count = ?,
                updates_applied = ?, updates_skipped = ?
            WHERE automation_run_id = ?;
            """,
            (
                datetime.now(timezone.utc).isoformat(),
                len(cases), excluded_count, saved_count,
                decision_counts["Auto Complete"],
                decision_counts["Agent Review"],
                decision_counts["Ambiguous Match"],
                len(applied_ids), len(skipped_ids), run_id,
            ),
        )

        db_connection.commit()

    except Exception as exc:
        db_connection.rollback()

        status = (
            "Rejected"
            if stage == "validation" and isinstance(exc, ValueError)
            else "Failed"
        )

        with db_connection:
            db_connection.execute(
                """
                UPDATE automation_runs
                SET run_status = ?, completed_at = ?, error_details = ?
                WHERE automation_run_id = ?;
                """,
                (
                    status,
                    datetime.now(timezone.utc).isoformat(),
                    str(exc),
                    run_id,
                ),
            )

        raise

    return summary()


print("Reusable accounting runner is ready.")

Reusable accounting runner is ready.


In [37]:
full_run_id = f"TEST-FULL-{uuid4().hex}"

full_run_result = run_accounting_check(
    test_accounting_path,
    ACCOUNTING_EXTRACT_DATE,
    connection,
    run_id=full_run_id,
)

full_run_result

,automation_run_id,run_status,eligible_case_count,completed_cases_excluded,cases_checked,exact_match_count,agent_review_count,ambiguous_match_count,updates_applied,updates_skipped,error_details
0,TEST-FULL-4cb3325d604c4ad4a95307c943deef64,Succeeded,2370,503,2370,0,2369,1,0,0,None


In [38]:
summary_row = full_run_result.iloc[0]

expected_counts = {
    "eligible_case_count": 2370,
    "completed_cases_excluded": 503,
    "cases_checked": 2370,
    "exact_match_count": 0,
    "agent_review_count": 2369,
    "ambiguous_match_count": 1,
    "updates_applied": 0,
    "updates_skipped": 0,
}

assert summary_row["run_status"] == "Succeeded"

for column, expected in expected_counts.items():
    assert summary_row[column] == expected, (
        f"{column}: expected {expected}, got {summary_row[column]}"
    )

tables_to_check = {
    "agent_updates": '"Case ID"',
    "automation_runs": "automation_run_id",
    "automation_results": "automation_run_id, case_id",
    "workflow_events": "event_id",
}

before_repeat = {
    table: pd.read_sql_query(
        f'SELECT * FROM "{table}" ORDER BY {order};',
        connection,
    )
    for table, order in tables_to_check.items()
}

repeat_result = run_accounting_check(
    test_accounting_path,
    ACCOUNTING_EXTRACT_DATE,
    connection,
    run_id=full_run_id,
)

pd.testing.assert_frame_equal(full_run_result, repeat_result)

for table, order in tables_to_check.items():
    pd.testing.assert_frame_equal(
        before_repeat[table],
        pd.read_sql_query(
            f'SELECT * FROM "{table}" ORDER BY {order};',
            connection,
        ),
    )

print("Full-run counts match expectations.")
print("Repeating the same run ID returns the saved result.")
print("No records, results or events changed.")

Full-run counts match expectations.
Repeating the same run ID returns the saved result.
No records, results or events changed.


In [39]:
fresh_accounting_path = (
    phase4_folder / "accounting_fresh_runner_test.db"
)

if fresh_accounting_path.exists():
    raise FileExistsError(
        "Fresh accounting test database already exists."
    )

with sqlite3.connect(
    phase3_database_path.as_uri() + "?mode=ro",
    uri=True,
) as baseline_connection:
    destination = sqlite3.connect(fresh_accounting_path)

    try:
        baseline_connection.backup(destination)
    finally:
        destination.close()

fresh_accounting_connection = sqlite3.connect(
    fresh_accounting_path.as_uri() + "?mode=rw",
    uri=True,
)

fresh_accounting_connection.execute("PRAGMA foreign_keys = ON;")

for table_name in [
    "automation_runs",
    "automation_results",
    "workflow_events",
]:
    table_sql = connection.execute(
        """
        SELECT sql FROM sqlite_master
        WHERE type = 'table' AND name = ?;
        """,
        (table_name,),
    ).fetchone()[0]

    fresh_accounting_connection.execute(table_sql)

fresh_accounting_connection.commit()

fresh_runner_before = pd.read_sql_query(
    'SELECT * FROM agent_updates ORDER BY "Case ID";',
    fresh_accounting_connection,
)

fresh_runner_id = f"TEST-FRESH-{uuid4().hex}"

fresh_runner_result = run_accounting_check(
    test_accounting_path,
    ACCOUNTING_EXTRACT_DATE,
    fresh_accounting_connection,
    run_id=fresh_runner_id,
)

expected_counts = {
    "eligible_case_count": 2372,
    "completed_cases_excluded": 501,
    "cases_checked": 2372,
    "exact_match_count": 2,
    "agent_review_count": 2369,
    "ambiguous_match_count": 1,
    "updates_applied": 2,
    "updates_skipped": 0,
}

assert fresh_runner_result.loc[0, "run_status"] == "Succeeded"

for column, expected in expected_counts.items():
    assert fresh_runner_result.loc[0, column] == expected, column

print("Fresh runner counts match expectations.")
print("Cases checked: 2,372")
print("Existing completions excluded: 501")
print("New automated completions applied: 2")

Fresh runner counts match expectations.
Cases checked: 2,372
Existing completions excluded: 501
New automated completions applied: 2


In [40]:
fresh_runner_after = pd.read_sql_query(
    'SELECT * FROM agent_updates ORDER BY "Case ID";',
    fresh_accounting_connection,
)

applied_results = pd.read_sql_query(
    """
    SELECT *
    FROM automation_results
    WHERE automation_run_id = ?
      AND case_update_applied = 1
    ORDER BY case_id;
    """,
    fresh_accounting_connection,
    params=(fresh_runner_id,),
)

completion_events = pd.read_sql_query(
    """
    SELECT *
    FROM workflow_events
    WHERE automation_run_id = ?
    ORDER BY case_id;
    """,
    fresh_accounting_connection,
    params=(fresh_runner_id,),
)

expected_case_ids = {"DH000001", "DH000008"}

assert set(applied_results["case_id"]) == expected_case_ids
assert len(applied_results) == 2
assert set(completion_events["case_id"]) == expected_case_ids
assert len(completion_events) == 2

expected_after = fresh_runner_before.copy()

expected_dates = {
    "DH000001": "2025-02-01",
    "DH000008": "2026-08-31",
}

for _, result in applied_results.iterrows():
    case_id = result["case_id"]

    assert result["decision"] == "Auto Complete"
    assert result["result_reason"] == "Exact Match"
    assert result["valid_match_count"] == 1
    assert result["completion_approved"] == 1
    assert result["refund_processed_date"] == expected_dates[case_id]

    event = completion_events.loc[
        completion_events["case_id"].eq(case_id)
    ].iloc[0]

    assert event["event_type"] == "Automated Completion"
    assert event["previous_status"] == "Open"
    assert event["new_status"] == "Completed"
    assert event["new_outcome"] == "Refund Already Processed"
    assert event["new_completed_by"] == "Automated Check"
    assert event["new_completion_date"] == expected_dates[case_id]
    assert event["event_timestamp"] == result["update_applied_at"]
    assert event["event_reason"] == (
        f"Exact Match: {result['matched_transaction_id']}"
    )

    changes = {
        "Case Status": "Completed",
        "Final Outcome": "Refund Already Processed",
        "Completed By": "Automated Check",
        "Completion Date": expected_dates[case_id],
        "Last Updated Date": result["update_applied_at"],
    }

    mask = expected_after["Case ID"].eq(case_id)

    for column, value in changes.items():
        expected_after[column] = expected_after[column].astype("object")
        expected_after.loc[mask, column] = value

pd.testing.assert_frame_equal(
    fresh_runner_after,
    expected_after,
    check_dtype=False,
    check_exact=True,
)

print("Only DH000001 and DH000008 were completed.")
print("Completion dates and both audit events are correct.")
print("All existing completions, assignments and notes are unchanged.")

Only DH000001 and DH000008 were completed.
Completion dates and both audit events are correct.
All existing completions, assignments and notes are unchanged.


In [41]:
duplicate_feed_path = (
    phase4_folder / "refund_transactions_duplicate_id.csv"
)

duplicate_feed = test_accounting_feed.copy()
duplicate_feed.loc[1, "Refund Transaction ID"] = (
    duplicate_feed.loc[0, "Refund Transaction ID"]
)

duplicate_feed.to_csv(duplicate_feed_path, index=False)

rejected_run_id = f"TEST-REJECTED-{uuid4().hex}"

tables_to_preserve = {
    "agent_updates": '"Case ID"',
    "automation_results": "automation_run_id, case_id",
    "workflow_events": "event_id",
}

before_rejection = {
    table: pd.read_sql_query(
        f'SELECT * FROM "{table}" ORDER BY {order};',
        fresh_accounting_connection,
    )
    for table, order in tables_to_preserve.items()
}

try:
    run_accounting_check(
        duplicate_feed_path,
        ACCOUNTING_EXTRACT_DATE,
        fresh_accounting_connection,
        run_id=rejected_run_id,
    )
except ValueError as exc:
    assert "Duplicate refund transaction IDs" in str(exc)
    print("Expected rejection:", exc)
else:
    raise AssertionError("Duplicate transaction IDs were not rejected.")

for table, order in tables_to_preserve.items():
    pd.testing.assert_frame_equal(
        before_rejection[table],
        pd.read_sql_query(
            f'SELECT * FROM "{table}" ORDER BY {order};',
            fresh_accounting_connection,
        ),
    )

rejected_log = pd.read_sql_query(
    """
    SELECT run_status, error_details
    FROM automation_runs
    WHERE automation_run_id = ?;
    """,
    fresh_accounting_connection,
    params=(rejected_run_id,),
)

assert len(rejected_log) == 1
assert rejected_log.loc[0, "run_status"] == "Rejected"

print("Rejected attempt retained in the run log.")
print("Operational records, results and events are unchanged.")

rejected_log

Expected rejection: Duplicate refund transaction IDs.
Rejected attempt retained in the run log.
Operational records, results and events are unchanged.


,run_status,error_details
0,Rejected,Duplicate refund transaction IDs.


In [42]:
failure_case = pd.read_sql_query(
    """
    SELECT source.*, work."Case Status"
    FROM source_cases AS source
    JOIN agent_updates AS work
        ON source."Case ID" = work."Case ID"
    WHERE source."Case ID" = 'DH000002';
    """,
    fresh_accounting_connection,
).iloc[0]

assert failure_case["Case Status"] != "Completed"

failure_feed_path = phase4_folder / "refund_transactions_failure_test.csv"

pd.DataFrame([{
    "Refund Transaction ID": "TEST-FAILURE-RT-001",
    "Policy Number": failure_case["Policy Number"],
    "Client Number": failure_case["Client Number"],
    "Refund Amount": str(
        Decimal(str(failure_case["Outstanding Amount"]))
    ),
    "Refund Processed By": "TEST-FIN001",
    "Refund Processed Date": (
        pd.Timestamp(failure_case["First Identified Date"])
        + pd.Timedelta(days=1)
    ).strftime("%Y-%m-%d"),
    "Refund Reason": "Synthetic runner rollback test",
}]).to_csv(failure_feed_path, index=False)

failed_run_id = f"TEST-FAILED-{uuid4().hex}"

before_failure = {
    table: pd.read_sql_query(
        f'SELECT * FROM "{table}" ORDER BY {order};',
        fresh_accounting_connection,
    )
    for table, order in tables_to_preserve.items()
}

try:
    fresh_accounting_connection.execute(
        """
        CREATE TEMP TRIGGER force_accounting_event_failure
        BEFORE INSERT ON main.workflow_events
        BEGIN
            SELECT RAISE(ABORT, 'Intentional event-write failure');
        END;
        """
    )
    fresh_accounting_connection.commit()

    try:
        run_accounting_check(
            failure_feed_path,
            ACCOUNTING_EXTRACT_DATE,
            fresh_accounting_connection,
            run_id=failed_run_id,
        )
    except sqlite3.IntegrityError as exc:
        assert "Intentional event-write failure" in str(exc)
        print("Expected processing failure:", exc)
    else:
        raise AssertionError("The intentional failure did not occur.")

finally:
    fresh_accounting_connection.rollback()
    fresh_accounting_connection.execute(
        "DROP TRIGGER IF EXISTS temp.force_accounting_event_failure;"
    )
    fresh_accounting_connection.commit()

for table, order in tables_to_preserve.items():
    pd.testing.assert_frame_equal(
        before_failure[table],
        pd.read_sql_query(
            f'SELECT * FROM "{table}" ORDER BY {order};',
            fresh_accounting_connection,
        ),
    )

failed_log = pd.read_sql_query(
    """
    SELECT run_status, error_details
    FROM automation_runs
    WHERE automation_run_id = ?;
    """,
    fresh_accounting_connection,
    params=(failed_run_id,),
)

assert len(failed_log) == 1
assert failed_log.loc[0, "run_status"] == "Failed"

print("Completion, results and events rolled back.")
print("Failed attempt retained in the run log.")
print("Temporary failure trigger removed.")

failed_log

Expected processing failure: Intentional event-write failure
Completion, results and events rolled back.
Failed attempt retained in the run log.
Temporary failure trigger removed.


,run_status,error_details
0,Failed,Intentional event-write failure
